In [15]:
import numpy as np

In [16]:
corpus = """knowing the name of something is different from knowing something
knowing something about everything is alright

"""

In [17]:
unique_words = np.unique(corpus.split())
print("unique words in corpus: ", np.unique(corpus.split()), "totaling: ", len(unique_words))

In [18]:
# Suppose that we use one hot encoding to convert the words in the sentences to vectors. Then each word in the sentences is converted to a vector of size?

one_hot_size = len(unique_words)


In [19]:
# Construct a co-occurrence matrix using the vocabulary developed in question 1. However, drop the following words from the vocabulary (and hence from the sentences): of, the, alright, about, from. Arrange the words in the vocabulary in alphabetical order. Use a window of size 1, k =1, How many non-zero entries are there in the co-occurrence matrix?

words = np.array([word for word in unique_words if word not in ["of", "the", "alright", "about", "from"]])
print("words: ", words)

In [20]:
# By using the cooccurence matrix created in Q3, compute the cosine similarity between the words in the vocabulary. While computing cosine similarity, ensure that each word vector is normalized to have a unit magnitude. The word knowing is closest to which of the following words?

cooccurence_matrix = np.zeros((len(words), len(words)))

for sentence in corpus.split("\n"):
    sentence = sentence.split()
    for i, word in enumerate(sentence):
        if word not in words:
            continue
        for j in range(max(0, i-1), min(len(sentence), i+2)):
            if i == j or sentence[j] not in words:
                continue
            cooccurence_matrix[np.where(words == word), np.where(words == sentence[j])] += 1

print("cooccurence matrix: \n", cooccurence_matrix)

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

cosine_similarities = np.zeros((len(words), len(words)))
for i in range(len(words)):
    for j in range(len(words)):
        cosine_similarities[i, j] = cosine_similarity(cooccurence_matrix[i], cooccurence_matrix[j])

print("cosine similarities: \n", cosine_similarities)

knowing_idx = np.where(words == "knowing")[0][0]
cosine_similarities[knowing_idx, knowing_idx] = 0
print("word knowing is closest to: ", words[np.argmax(cosine_similarities[knowing_idx])])


In [21]:
# Compute the Pointwise Mutual Information (PMI) for the pair (knowing,something). Take N=9 and log to the base 2. (Enter the answer upto 3 decimal places)

N = 9
knowing_idx = np.where(words == "knowing")[0][0]
something_idx = np.where(words == "something")[0][0]
p_knowing = np.sum(cooccurence_matrix[knowing_idx]) / N
p_something = np.sum(cooccurence_matrix[something_idx]) / N
p_knowing_something = cooccurence_matrix[knowing_idx, something_idx] / N
pmi = np.log2(p_knowing_something / (p_knowing * p_something))
print("pmi: ", np.round(pmi, 3))


In [22]:
# Calculate the PPMI for Q5 and enter the value upto 3 decimal points

ppmi = max(pmi, 0)
print("ppmi: ", np.round(ppmi, 3))


In [23]:
# Compute the SVD of the (normalized) co-occurrence matrix and take the rank-1 approximation (round the values in the matrix to 3 decimal points). Which of the following words are closer to the word knowing? We say the word is closer to the word \textbf{knowing} if its similarity score is greater than 0.5.

u, s, vh = np.linalg.svd(cooccurence_matrix, full_matrices=False)
rank_1_approximation = np.round(np.outer(u[:, 0], np.outer(s[0], vh[0])), 3)
print("rank 1 approximation: \n", rank_1_approximation)
knowing_idx = np.where(words == "knowing")[0][0]
for i in range(len(words)):
    if i == knowing_idx:
        continue
    norm1 = np.linalg.norm(rank_1_approximation[knowing_idx])
    norm2 = np.linalg.norm(rank_1_approximation[i])
    similarity = np.dot(rank_1_approximation[knowing_idx], rank_1_approximation[i]) / (norm1 * norm2) if norm1 and norm2 else 0
    if similarity > 0.5:
        print("word closer to knowing: ", words[i])



In [24]:
# CBOW model parameters calculation
vocabulary_size = 10000
vector_size = 100

# For CBOW, we have two weight matrices:
# 1. Input->Hidden (W): vocabulary_size x vector_size
# 2. Hidden->Output (W'): vector_size x vocabulary_size

# For Wword (Hidden->Output matrix), the parameters are:
params = vector_size * vocabulary_size

# Convert to thousands
params_in_thousands = params / 1000
print(f"Number of parameters in Wword: {params_in_thousands:.0f}K")